<a href="https://colab.research.google.com/github/sriharsha0092/Tetris/blob/master/hampi_multisite_pipeline_(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Satellite site-detection: Hampi baseline + multi-site test + matched random reference points


| # | Section | What it does |
|---|---------|--------------|
| 1 | Setup | imports, Earth Engine auth |
| 2 | Settings | every threshold and seed, frozen in one place |
| 3 | Functions | cloud mask, features, sampling (cached), statistics |
| 4 | Map check | look at the circle and ring before reading any number |
| 5 | Round 1 | whole-circle vs ring, natural land only |
| 6 | Round 2 | same comparison *within* land-cover class (shrub / grass / bare) |
| 7 | Reference points | random places 5-20 km away with matched radius: how big is |d| by chance? |
| 8 | PostGIS | load every sampled pixel, indexed, 3 spatial queries, `pg_dump` |
| 9 | Export | CSVs + `settings_record.json` + zip |

**What this can and cannot show.** Hampi is a null result: after land-cover control the site/ring difference is small (|d| <= ~0.3). A null cannot show the method *finds* unknown sites. The multi-site rows are exploratory (hypotheses, not findings); section 7 shows how large |d| gets at random places, which is the yardstick for reading them.

### Amendment log (read this before anything else)
Order in which the analysis changed, and why. Nothing below was tuned to make an effect appear.
1. **Round 1** (whole circle vs ring) was run first on five sites.
2. Round 1 showed large |d| at several sites, mostly in elevation, slope, seasonal NDVI and radar. These are what a *land-cover difference* looks like, so **round 2** (within-class) was added *after* seeing round 1.
3. **Warangal_central excluded from round 2** because the circle is urban (about 8% of the circle survived the natural-land mask). Reason is land cover, not effect size. It stays visible in round 1.
4. The first round-2 code returned identical rows for all classes (a bug: pooled, not stratified). It was replaced by sampling once and splitting by class in pandas. The identical output was discarded.
5. **Hire_Benakal**: the circle sits on the village, not on the tomb cluster. Flagged in every results table; treat its rows as unreliable.
6. First reference points (seed 2026, fixed 200 m radius) were thin and unmatched. **Superseded** by the matched design here (seed 2027, radius matched to the parent site). The old numbers are not used.
7. Ring radii (300-1200 m), thresholds and sample size were not changed between rounds.
8. 8 of 18 reference points were unusable (too few natural-land pixels after the mask). Seed not changed.
9. Earlier ad-hoc round-1 numbers (different ring/sampling) are superseded by this frozen design.
10. **Cloud mask replaced.** The original QA60 mask stops working on 2022-01-25 (Earth Engine lists QA60 as masked out until 2024-02-28), so the composites in the first runs are not trustworthy for 2022-2023 (those images were either dropped or left unmasked; section 3b shows which). The mask was replaced with Google Cloud Score+ (threshold 0.60) on data-quality grounds, and everything was re-run. Results from the legacy mask are superseded.
11. First PostGIS load failed silently (lc written as float); fixed by casting lc to integer before export. No analysis numbers were affected.

Analyses run: 2 rounds on 5 sites x 10 features, plus the reference batch. Tests are two-sided Mann-Whitney U; Cohen's d (pooled SD) is the effect size. With thousands of spatially autocorrelated pixels p-values are meaningless; read d.

## 1. Setup

In [ ]:
import json, os, sys, time, math, subprocess, zipfile, datetime
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
import ee, geemap

# >>> put your Google Cloud project id here
EE_PROJECT = 'YOUR-PROJECT-ID'

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print('earth engine ready')

earth engine ready


## 2. Settings (frozen)
Change nothing here after you have seen results. Everything the analysis depends on is in this cell, and it is written to `settings_record.json` at the end.

In [ ]:
START, END = '2021-01-01', '2023-06-06'
MASK_MODE = 'cs_plus'                  # 'cs_plus' = Google Cloud Score+ (use this); 'qa60' = legacy mask, BROKEN 2022-01-25 to 2024-02-28
CS_THRESH = 0.60                       # keep pixels with cs_cdf >= this (clear)
OUT = f'outputs_{MASK_MODE}'           # separate output folder per mask mode
os.makedirs(OUT, exist_ok=True)
RING_IN, RING_OUT = 500, 1000          # ring = annulus between these radii (m) around the site centre
N_SAMPLE = 4000                        # pixels requested per zone
SCALE = 10                             # m
SAMPLE_SEED = 42

MIN_N_SITE, MIN_N_RING = 100, 300      # round 1 minimum usable pixels
MIN_N_CLASS = 60                       # round 2 minimum per class, both zones
GREEN, DRY = (10, 12), (2, 4)          # months for the seasonal NDVI difference
MIN_IMAGES = 5                         # minimum images in a seasonal window

CLASSES = {20: 'shrub', 30: 'grass', 60: 'bare'}   # ESA WorldCover v100 'natural' classes
NATURAL = list(CLASSES.keys())

# matched random reference points
REF_SEED = 2027
REF_PER_SITE = 6
REF_SITES = ['Hampi_baseline', 'Pattadakal', 'Ramappa']
REF_DIST = (5000, 20000)               # m from the parent site
MIN_REFS = 8                           # a feature needs at least this many usable refs to be judged

# sites: name -> (lon, lat, radius_m)
SITES = {
    'Hampi_baseline':   (76.468536, 15.334022, 300),
    'Pattadakal':       (75.81561,  15.95019,  300),
    'Ramappa':          (79.94333,  18.25917,  300),
    'Hire_Benakal':     (76.4560,   15.4378,   300),
    'Warangal_central': (79.616053, 17.956286, 300),
}
# honest flags carried into every results table
FLAGS = {
    'Hire_Benakal': 'circle on village, not tomb cluster',
    'Warangal_central': 'urban',
}
EXCLUDE_R2 = {'Warangal_central': 'urban; ~8% of circle survived the natural-land mask (decided after round 1)'}

FEATURES = ['NDVI', 'SAVI', 'NDMI', 'BSI', 'SWIR1', 'NDVI_green_minus_dry', 'VV', 'VH', 'elev', 'slope']
LOCS = dict(SITES)      # sites + (later) reference points
print(len(SITES), 'sites;', len(FEATURES), 'features')

5 sites; 10 features


## 3. Functions

In [ ]:
def mask_s2_clouds(image):
    """LEGACY. QA60 bit 10 = opaque cloud, bit 11 = cirrus. QA60 is masked out for 2022-01-25 to 2024-02-28
    (ESA stopped producing the polygons), so this mask is unusable for most of 2022-2023. Kept only for comparison."""
    qa = image.select('QA60')
    clear = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return (image.updateMask(clear).divide(10000)
            .copyProperties(image, ['system:time_start']))

def mask_cs_plus(image):
    """Cloud Score+ mask: cs_cdf is the 0-1 clear-sky probability. Keep the 4 bands used and scale to reflectance."""
    clear = image.select('cs_cdf').gte(CS_THRESH)
    return (image.select(['B2', 'B4', 'B8', 'B11']).updateMask(clear).divide(10000)
            .copyProperties(image, ['system:time_start']))

def prep_s2(region, mode=None):
    """Masked Sentinel-2 SR collection for the region, using the chosen mask mode."""
    mode = mode or MASK_MODE
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(region).filterDate(START, END))
    if mode == 'cs_plus':
        cs = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
        return col.linkCollection(cs, ['cs_cdf']).map(mask_cs_plus)
    return col.map(mask_s2_clouds)

def cohens_d(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    n1, n2 = len(a), len(b)
    sp = math.sqrt(((n1 - 1) * a.var(ddof=1) + (n2 - 1) * b.var(ddof=1)) / (n1 + n2 - 2))
    return (a.mean() - b.mean()) / sp if sp > 0 else np.nan

def zone_geoms(lon, lat, r):
    c = ee.Geometry.Point([lon, lat])
    site = c.buffer(r)
    ring = c.buffer(RING_OUT).difference(c.buffer(RING_IN))
    return c, site, ring

def season_ndvi(s2, months):
    sub = s2.filter(ee.Filter.calendarRange(months[0], months[1], 'month'))
    n = sub.size().getInfo()
    if n < MIN_IMAGES:
        return None, n
    return sub.median().normalizedDifference(['B8', 'B4']), n

def build_features(lon, lat):
    """Builds the feature image. The composite is rebuilt from the collection every call (no stale objects)."""
    region = ee.Geometry.Point([lon, lat]).buffer(RING_OUT)
    s2 = prep_s2(region)
    comp = s2.median()
    B2, B4, B8, B11 = comp.select('B2'), comp.select('B4'), comp.select('B8'), comp.select('B11')

    ndvi = comp.normalizedDifference(['B8', 'B4']).rename('NDVI')
    savi = B8.subtract(B4).divide(B8.add(B4).add(0.5)).multiply(1.5).rename('SAVI')
    ndmi = comp.normalizedDifference(['B8', 'B11']).rename('NDMI')
    bsi = (B11.add(B4).subtract(B8.add(B2))).divide(B11.add(B4).add(B8.add(B2))).rename('BSI')
    swir1 = B11.rename('SWIR1')

    g, ng = season_ndvi(s2, GREEN)
    d, nd = season_ndvi(s2, DRY)
    if g is None or d is None:
        print(f'   seasonal window too thin (green={ng}, dry={nd} images): feature skipped')
        seas = ee.Image.constant(0).updateMask(ee.Image.constant(0)).rename('NDVI_green_minus_dry')
    else:
        seas = g.subtract(d).rename('NDVI_green_minus_dry')

    s1 = (ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(region).filterDate(START, END)
          .filter(ee.Filter.eq('instrumentMode', 'IW'))
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
          .select(['VV', 'VH']).median())

    elev = ee.Image('USGS/SRTMGL1_003').rename('elev')
    slope = ee.Terrain.slope(ee.Image('USGS/SRTMGL1_003')).rename('slope')

    lc = ee.ImageCollection('ESA/WorldCover/v100').mosaic().select('Map').rename('lc')
    natural = lc.eq(20).Or(lc.eq(30)).Or(lc.eq(60))

    img = ee.Image.cat([ndvi, savi, ndmi, bsi, swir1, seas, s1, elev, slope, lc])
    return img.updateMask(natural)

def sample_zone(img, geom):
    fc = img.sample(region=geom, scale=SCALE, numPixels=N_SAMPLE, seed=SAMPLE_SEED,
                    geometries=True, tileScale=4)
    rows = []
    for f in fc.getInfo()['features']:
        p = dict(f['properties'])
        p['lon'], p['lat'] = f['geometry']['coordinates']
        rows.append(p)
    df = pd.DataFrame(rows)
    return df.reindex(columns=FEATURES + ['lc', 'lon', 'lat'])

_ZONES = {}
def get_zones(name):
    """One sampling call per zone per location; cached on (name, lon, lat, r). Round 2 splits these by class in pandas."""
    lon, lat, r = LOCS[name]
    key = (name, lon, lat, r)
    if key not in _ZONES:
        img = build_features(lon, lat)
        _, site, ring = zone_geoms(lon, lat, r)
        s = sample_zone(img, site); s['zone'] = 'site'
        g = sample_zone(img, ring); g['zone'] = 'ring'
        df = pd.concat([s, g], ignore_index=True)
        df.insert(0, 'site', name)
        _ZONES[key] = df
    return _ZONES[key]

def compare(a, b, label, extra):
    rows = []

    for f in FEATURES:
        x, y = a[f].dropna().values, b[f].dropna().values
        if len(x) < 2 or len(y) < 2:
            continue
        p = mannwhitneyu(x, y, alternative='two-sided').pvalue
        rows.append(dict(site=label, **extra, feature=f, n_site=len(x), n_ring=len(y),
                         site_mean=x.mean(), ring_mean=y.mean(), p=p, d=cohens_d(x, y),jm=jm(x, y),
                         note=FLAGS.get(label, '')))
    return rows

def run_round1(names, verbose=True):
    rows = []
    for name in names:
        df = get_zones(name)
        s, g = df[df.zone == 'site'], df[df.zone == 'ring']
        if verbose:
            print(f'  {name}: site n={len(s)}, ring n={len(g)}')
        if len(s) < MIN_N_SITE or len(g) < MIN_N_RING:
            if verbose: print('     skipped (too few pixels after mask)')
            continue
        rows += compare(s, g, name, {})
    return pd.DataFrame(rows)

def run_round2(names, verbose=True):
    rows = []
    for name in names:
        if name in EXCLUDE_R2:
            if verbose: print(f'  {name}: excluded - {EXCLUDE_R2[name]}')
            continue
        df = get_zones(name)
        for code_, cls in CLASSES.items():
            sub = df[df.lc == code_]
            s, g = sub[sub.zone == 'site'], sub[sub.zone == 'ring']
            if verbose:
                print(f'  {name} / {cls}: site n={len(s)}, ring n={len(g)}')
            if len(s) < MIN_N_CLASS or len(g) < MIN_N_CLASS:
                continue
            rows += compare(s, g, name, {'cls': cls})
    return pd.DataFrame(rows)

def show(name):
    lon, lat, r = LOCS[name]
    c, site, ring = zone_geoms(lon, lat, r)
    m = geemap.Map()
    m.add_basemap('SATELLITE')
    m.centerObject(c, 15)
    m.addLayer(ring, {'color': 'blue'}, 'ring')
    m.addLayer(site, {'color': 'red'}, 'site')
    return m

def make_refs():
    """Random reference points: same radius as the parent site, 5-20 km away, fixed seed."""
    rng = np.random.default_rng(REF_SEED)
    refs = {}
    for parent in REF_SITES:
        lon0, lat0, r = SITES[parent]
        for i in range(REF_PER_SITE):
            ang = rng.uniform(0, 2 * np.pi)
            dist = rng.uniform(*REF_DIST)
            dlat = dist * np.sin(ang) / 111000.0
            dlon = dist * np.cos(ang) / (111000.0 * np.cos(np.radians(lat0)))
            refs[f'ref_{parent}_{i+1}'] = (lon0 + dlon, lat0 + dlat, r)
    return refs

def ref_summary(site_r1, ref_r1):
    ra = ref_r1.assign(absd=ref_r1.d.abs()).groupby('feature').absd.agg(
        ref_n='count', ref_median='median', ref_p90=lambda s: s.quantile(0.9))
    sa = site_r1.assign(site_absd=site_r1.d.abs())[['site', 'feature', 'd', 'site_absd', 'note']]
    m = sa.merge(ra, left_on='feature', right_index=True, how='left')
    m['verdict'] = np.where(m.ref_n.fillna(0) < MIN_REFS, 'too few refs',
                            np.where(m.site_absd > m.ref_p90, 'above ref p90', 'within ref range'))
    return m
print('functions defined')

functions defined


In [ ]:
def jm(a, b):
    """Jeffries-Matusita distance, 0 (identical) to 2 (fully separable). Assumes roughly Gaussian values."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    m1, m2, v1, v2 = a.mean(), b.mean(), a.var(ddof=1), b.var(ddof=1)
    B = 0.25*(m1-m2)**2/(v1+v2) + 0.5*np.log((v1+v2)/(2*np.sqrt(v1*v2)))
    return 2*(1-np.exp(-B))

## 3b. Mask check: how much of the circle survives the cloud mask, by month
The legacy QA60 mask stops working on 2022-01-25 (Earth Engine documents QA60 as masked out until 2024-02-28). This compares the two masks on the Hampi circle. If `valid_qa60` drops to about 0 from 2022-02 onward while `valid_cs_plus` stays above 0, the old mask was discarding every image after that date.

In [ ]:
def valid_by_month(lon, lat, r, mode):
    """Per-image fraction of the site circle left unmasked, averaged by month."""
    region = ee.Geometry.Point([lon, lat]).buffer(RING_OUT)
    site = ee.Geometry.Point([lon, lat]).buffer(r)
    col = prep_s2(region, mode)
    def frac(img):
        v = img.select('B8').mask().reduceRegion(ee.Reducer.mean(), site, 20, maxPixels=1e8)
        return ee.Feature(None, {'month': img.date().format('YYYY-MM'), 'valid': v.get('B8')})
    rows = [f['properties'] for f in col.map(frac).getInfo()['features']]
    df = pd.DataFrame(rows)
    return df.groupby('month').valid.agg(n_images='count', **{f'valid_{mode}': 'mean'})

lon0, lat0, r0 = SITES['Hampi_baseline']
mask_check = (valid_by_month(lon0, lat0, r0, 'qa60')
              .join(valid_by_month(lon0, lat0, r0, 'cs_plus')[['valid_cs_plus']], how='outer')).round(2)
print(mask_check.to_string())
mask_check.to_csv(f'{OUT}/mask_check_by_month.csv')

         n_images  valid_qa60  valid_cs_plus
month                                       
2021-01        14        0.52           0.71
2021-02        10        1.00           1.00
2021-03        12        1.00           1.00
2021-04        12        1.00           1.00
2021-05        14        0.86           0.86
2021-06        12        0.50           0.33
2021-07        12        0.00           0.00
2021-08        12        0.06           0.17
2021-09        12        0.23           0.31
2021-10        12        0.53           0.69
2021-11        12        0.33           0.33
2021-12        12        0.67           0.67
2022-01        14        0.71           0.84
2022-02        10        1.00           1.00
2022-03        12        1.00           1.00
2022-04        14        1.00           0.71
2022-05        14        1.00           0.61
2022-06        12        1.00           0.33
2022-07        12        1.00           0.15
2022-08        12        1.00           0.13
2022-09   

## 4. Map check
Look at the red circle (site) and blue ring before reading numbers. Re-run with other site names. `Hire_Benakal` should visibly sit on the village; if you relocate it, edit `SITES` in section 2 *before* running anything and add a line to the amendment log.

In [ ]:
show('Hampi_baseline')

## 5. Round 1 - whole circle vs ring (natural land only)

In [ ]:
ALL_SITES = list(SITES.keys())
r1 = run_round1(ALL_SITES)
r1_view = r1.copy()
for c in ['site_mean', 'ring_mean', 'd']:
    r1_view[c] = r1_view[c].round(3)
r1_view['p'] = r1_view['p'].round(3)
print(r1_view.to_string(index=False))
print()
print(r1.pivot(index='feature', columns='site', values='d').round(2).to_string())

  Hampi_baseline: site n=2381, ring n=2209
  Pattadakal: site n=539, ring n=399
  Ramappa: site n=277, ring n=369
  Hire_Benakal: site n=419, ring n=500
  Warangal_central: site n=250, ring n=262
     skipped (too few pixels after mask)
          site              feature  n_site  n_ring  site_mean  ring_mean     p      d       jm                                note
Hampi_baseline                 NDVI    2381    2209      0.284      0.279 0.000  0.043 0.015628                                    
Hampi_baseline                 SAVI    2381    2209      0.171      0.161 0.000  0.155 0.019047                                    
Hampi_baseline                 NDMI    2381    2209     -0.057     -0.068 0.000  0.170 0.043560                                    
Hampi_baseline                  BSI    2381    2209      0.110      0.116 0.000 -0.118 0.043466                                    
Hampi_baseline                SWIR1    2381    2209      0.242      0.233 0.000  0.236 0.027393        

## 6. Round 2 - within land-cover class
Same pixels as round 1 (cached, not resampled), split by WorldCover class so a site is compared with ring pixels of the *same* class. Cells with fewer than `MIN_N_CLASS` pixels in either zone are left blank rather than estimated.

In [ ]:
r2 = run_round2(ALL_SITES)
print()
print(r2.pivot_table(index=['feature', 'cls'], columns='site', values='jm').round(2).to_string())

  Hampi_baseline / shrub: site n=1018, ring n=870
  Hampi_baseline / grass: site n=34, ring n=60
  Hampi_baseline / bare: site n=1329, ring n=1279
  Pattadakal / shrub: site n=183, ring n=218
  Pattadakal / grass: site n=124, ring n=34
  Pattadakal / bare: site n=232, ring n=147
  Ramappa / shrub: site n=0, ring n=12
  Ramappa / grass: site n=105, ring n=246
  Ramappa / bare: site n=172, ring n=111
  Hire_Benakal / shrub: site n=35, ring n=185
  Hire_Benakal / grass: site n=27, ring n=22
  Hire_Benakal / bare: site n=357, ring n=293
  Warangal_central: excluded - urban; ~8% of circle survived the natural-land mask (decided after round 1)

site                        Hampi_baseline  Hire_Benakal  Pattadakal  Ramappa
feature              cls                                                     
BSI                  bare             0.05          0.09        0.04     0.00
                     grass             NaN           NaN         NaN     0.35
                     shrub            0.0

## 7. Matched random reference points
Each reference point copies its parent site's radius and gets the same ring, so it is a "site" with nothing known there. `ref_p90` is the |d| that 90% of random places stay under; a real site only stands out if it is above that, and even then it is a hypothesis. Features with fewer than `MIN_REFS` usable reference points are not judged.

In [ ]:
REFS = make_refs()
LOCS.update(REFS)
print(len(REFS), 'reference points, seed', REF_SEED)
ref_r1 = run_round1(list(REFS.keys()), verbose=False)
print('usable reference points:', ref_r1.site.nunique(), 'of', len(REFS))
refsum = ref_summary(r1[r1.site.isin(REF_SITES)], ref_r1)
view = refsum.copy()
for c in ['d', 'site_absd', 'ref_median', 'ref_p90']:
    view[c] = view[c].round(2)
print(view.sort_values(['site', 'feature']).to_string(index=False))

18 reference points, seed 2027
usable reference points: 14 of 18
          site              feature     d  site_absd note  ref_n  ref_median  ref_p90          verdict
Hampi_baseline                  BSI -0.12       0.12          14        0.27     0.72 within ref range
Hampi_baseline                 NDMI  0.17       0.17          14        0.30     0.80 within ref range
Hampi_baseline                 NDVI  0.04       0.04          14        0.33     0.58 within ref range
Hampi_baseline NDVI_green_minus_dry  0.24       0.24          14        0.32     1.32 within ref range
Hampi_baseline                 SAVI  0.16       0.16          14        0.37     0.56 within ref range
Hampi_baseline                SWIR1  0.24       0.24          14        0.32     0.76 within ref range
Hampi_baseline                   VH -0.05       0.05          14        0.32     0.50 within ref range
Hampi_baseline                   VV -0.03       0.03          14        0.28     0.57 within ref range
Hampi_ba

In [ ]:
ref_r2 = run_round2(list(REFS.keys()), verbose=False)

In [ ]:
def ref_summary_r2(site_r2, ref_r2, col='d'):
    ra = (ref_r2.assign(a=ref_r2[col].abs()).groupby(['feature','cls']).a
          .agg(ref_n='count', ref_median='median', ref_p90=lambda s: s.quantile(0.9)).reset_index())
    sa = site_r2.assign(site_abs=site_r2[col].abs())[['site','cls','feature',col,'site_abs','note']]
    m = sa.merge(ra, on=['feature','cls'], how='left')
    m['verdict'] = np.where(m.ref_n.fillna(0) < MIN_REFS, 'too few refs',
                    np.where(m.site_abs > m.ref_p90, 'above ref p90', 'within ref range'))
    return m

r2_sum = ref_summary_r2(r2[r2.site.isin(REF_SITES)], ref_r2)           # for d
r2_sum_jm = ref_summary_r2(r2[r2.site.isin(REF_SITES)], ref_r2, 'jm')  # for JM
print(r2_sum.sort_values(['site','cls','feature']).round(2).to_string(index=False))

          site   cls              feature     d  site_abs note  ref_n  ref_median  ref_p90          verdict
Hampi_baseline  bare                  BSI -0.22      0.22           7        0.52     1.06     too few refs
Hampi_baseline  bare                 NDMI  0.24      0.24           7        0.40     1.02     too few refs
Hampi_baseline  bare                 NDVI  0.09      0.09           7        0.16     0.53     too few refs
Hampi_baseline  bare NDVI_green_minus_dry  0.32      0.32           7        0.23     0.64     too few refs
Hampi_baseline  bare                 SAVI  0.19      0.19           7        0.20     0.51     too few refs
Hampi_baseline  bare                SWIR1  0.08      0.08           7        0.33     0.65     too few refs
Hampi_baseline  bare                   VH  0.01      0.01           7        0.09     0.45     too few refs
Hampi_baseline  bare                   VV  0.09      0.09           7        0.18     0.40     too few refs
Hampi_baseline  bare        

## 8. PostGIS
Loads every sampled pixel (sites, rings and reference points) with its coordinates into PostgreSQL/PostGIS, builds a GIST index, and runs three spatial queries. Colab/Linux with root. First run installs packages (a few minutes).

In [ ]:
def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, stdin=subprocess.DEVNULL)
    if check and r.returncode != 0:
        print(r.stdout, r.stderr)
    return r

sh('apt-get -qq update && apt-get -qq install -y postgresql postgresql-contrib postgis')
sh('service postgresql start'); time.sleep(6)
sh('''su postgres -c "psql -w -c 'CREATE DATABASE hampi'"''', check=False)   # ok if it already exists

def psql(sql, name='q'):
    path = f'/tmp/{name}.sql'
    open(path, 'w').write(sql); os.chmod(path, 0o644)
    r = sh(f'su postgres -c "psql -w -d hampi -f {path}"')
    return r.stdout + r.stderr      # show errors: psql -f keeps going after a failed statement

# all sampled pixels -> one CSV
allpix = pd.concat(_ZONES.values(), ignore_index=True)
allpix.columns = [c.lower() for c in allpix.columns]
allpix = allpix.rename(columns={'ndvi_green_minus_dry': 'ndvi_seasonal'})
cols = ['site', 'zone', 'lc', 'lon', 'lat', 'ndvi', 'savi', 'ndmi', 'bsi', 'swir1',
        'ndvi_seasonal', 'vv', 'vh', 'elev', 'slope']
allpix['lc'] = allpix['lc'].astype('Int64')     # NaN makes lc float ('20.0'), which Postgres rejects for an int column
allpix[cols].to_csv('/tmp/pixels.csv', index=False, na_rep='')
os.chmod('/tmp/pixels.csv', 0o644)

centres = pd.DataFrame([(k[0], k[1], k[2], k[3]) for k in _ZONES.keys()],
                       columns=['site', 'lon', 'lat', 'radius_m'])
centres.to_csv('/tmp/centres.csv', index=False)
os.chmod('/tmp/centres.csv', 0o644)

print(psql('''
CREATE EXTENSION IF NOT EXISTS postgis;
DROP TABLE IF EXISTS ndvi_samples;  DROP TABLE IF EXISTS site_centres;
CREATE TABLE ndvi_samples (
  id serial PRIMARY KEY, site text, zone text, lc int, lon double precision, lat double precision,
  ndvi real, savi real, ndmi real, bsi real, swir1 real, ndvi_seasonal real,
  vv real, vh real, elev real, slope real, geom geometry(Point, 4326));
CREATE TABLE site_centres (site text PRIMARY KEY, lon double precision, lat double precision,
  radius_m int, geom geometry(Point, 4326));
\\copy ndvi_samples (site,zone,lc,lon,lat,ndvi,savi,ndmi,bsi,swir1,ndvi_seasonal,vv,vh,elev,slope) FROM '/tmp/pixels.csv' CSV HEADER
\\copy site_centres (site,lon,lat,radius_m) FROM '/tmp/centres.csv' CSV HEADER
UPDATE ndvi_samples SET geom = ST_SetSRID(ST_MakePoint(lon, lat), 4326);
UPDATE site_centres SET geom = ST_SetSRID(ST_MakePoint(lon, lat), 4326);
CREATE INDEX ndvi_samples_geom_idx ON ndvi_samples USING GIST (geom);
ANALYZE ndvi_samples;
SELECT count(*) AS rows_loaded FROM ndvi_samples;
''', 'load'))

CREATE EXTENSION
DROP TABLE
DROP TABLE
CREATE TABLE
CREATE TABLE
COPY 61852
COPY 23
UPDATE 61852
UPDATE 23
CREATE INDEX
ANALYZE
 rows_loaded 
-------------
       61852
(1 row)

psql:/tmp/load.sql:3: NOTICE:  table "ndvi_samples" does not exist, skipping
psql:/tmp/load.sql:3: NOTICE:  table "site_centres" does not exist, skipping



In [ ]:
print('QUERY 1: mean NDVI and pixel count by site and zone')
print(psql('''
SELECT site, zone, count(*) AS n, round(avg(ndvi)::numeric, 3) AS mean_ndvi
FROM ndvi_samples WHERE site NOT LIKE 'ref_%' GROUP BY site, zone ORDER BY site, zone;
''', 'q1'))

print('QUERY 2: Hampi mean NDVI in 100 m distance bands from the true site centre')
print(psql('''
SELECT (floor(ST_Distance(s.geom::geography, c.geom::geography) / 100) * 100)::int AS band_start_m,
       count(*) AS n, round(avg(s.ndvi)::numeric, 3) AS mean_ndvi
FROM ndvi_samples s JOIN site_centres c ON c.site = s.site
WHERE s.site = 'Hampi_baseline'
GROUP BY 1 ORDER BY 1;
''', 'q2'))

print('QUERY 3: land-cover mix by site and zone (the confound, in SQL)')
print(psql('''
SELECT site, zone,
  round(100.0 * avg((lc = 20)::int), 1) AS pct_shrub,
  round(100.0 * avg((lc = 30)::int), 1) AS pct_grass,
  round(100.0 * avg((lc = 60)::int), 1) AS pct_bare
FROM ndvi_samples WHERE site NOT LIKE 'ref_%' GROUP BY site, zone ORDER BY site, zone;
''', 'q3'))

sh(f'su postgres -c "pg_dump -w hampi" > {OUT}/hampi_postgis_dump.sql')
print('pg_dump ->', OUT + '/hampi_postgis_dump.sql')

QUERY 1: mean NDVI and pixel count by site and zone
       site       | zone |  n   | mean_ndvi 
------------------+------+------+-----------
 Hampi_baseline   | ring | 2209 |     0.279
 Hampi_baseline   | site | 2381 |     0.284
 Hire_Benakal     | ring |  500 |     0.265
 Hire_Benakal     | site |  419 |     0.234
 Pattadakal       | ring |  399 |     0.307
 Pattadakal       | site |  539 |     0.372
 Ramappa          | ring |  369 |     0.406
 Ramappa          | site |  277 |     0.406
 Warangal_central | ring |  262 |     0.368
 Warangal_central | site |  250 |     0.304
(10 rows)


QUERY 2: Hampi mean NDVI in 100 m distance bands from the true site centre
 band_start_m |  n   | mean_ndvi 
--------------+------+-----------
            0 |  326 |     0.293
          100 |  893 |     0.283
          200 | 1162 |     0.283
          400 |   16 |     0.260
          500 |  328 |     0.234
          600 |  385 |     0.300
          700 |  401 |     0.286
          800 |  538 |     0.286

## 9. Export (the evidence pack)

In [ ]:
r1.to_csv(f'{OUT}/round1_results.csv', index=False)
r2.to_csv(f'{OUT}/round2_results.csv', index=False)
refsum.to_csv(f'{OUT}/reference_point_summary.csv', index=False)
allpix.to_csv(f'{OUT}/all_sampled_pixels.csv', index=False)

record = dict(
    run_time_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    ee_version=ee.__version__, geemap_version=geemap.__version__,
    START=START, END=END, MASK_MODE=MASK_MODE, CS_THRESH=CS_THRESH, RING_IN=RING_IN, RING_OUT=RING_OUT, N_SAMPLE=N_SAMPLE, SCALE=SCALE,
    SAMPLE_SEED=SAMPLE_SEED, MIN_N_SITE=MIN_N_SITE, MIN_N_RING=MIN_N_RING, MIN_N_CLASS=MIN_N_CLASS,
    GREEN=GREEN, DRY=DRY, MIN_IMAGES=MIN_IMAGES, CLASSES=CLASSES,
    REF_SEED=REF_SEED, REF_PER_SITE=REF_PER_SITE, REF_SITES=REF_SITES, REF_DIST=REF_DIST, MIN_REFS=MIN_REFS,
    SITES=SITES, FLAGS=FLAGS, EXCLUDE_R2=EXCLUDE_R2, REFS=REFS, FEATURES=FEATURES,
)
json.dump(record, open(f'{OUT}/settings_record.json', 'w'), indent=2, default=str)

with zipfile.ZipFile('evidence_pack.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(OUT):
        z.write(os.path.join(OUT, fn), fn)
print('wrote evidence_pack.zip:', sorted(os.listdir(OUT)))

wrote evidence_pack.zip: ['all_sampled_pixels.csv', 'hampi_postgis_dump.sql', 'mask_check_by_month.csv', 'reference_point_summary.csv', 'round1_results.csv', 'round2_results.csv', 'settings_record.json']


In [ ]:
def dist_m(df, lon0, lat0):
    dx = (df.lon - lon0) * 111320 * np.cos(np.radians(lat0))
    dy = (df.lat - lat0) * 110540
    return np.sqrt(dx**2 + dy**2)

RADII = [100, 150, 200, 300]
rows = []
for name in list(REF_SITES) + list(REFS.keys()):          # the 3 real sites + their matched random places
    lon0, lat0, _ = LOCS[name]
    df = get_zones(name)                                   # cached: no new Earth Engine calls if the session is alive
    s, g = df[df.zone == "site"].copy(), df[df.zone == "ring"]
    s["dist"] = dist_m(s, lon0, lat0)
    for r in RADII:
        sr = s[s.dist <= r]
        if len(sr) < MIN_N_SITE or len(g) < MIN_N_RING:
            continue
        for f in FEATURES:
            x, y = sr[f].dropna().values, g[f].dropna().values
            if len(x) > 1 and len(y) > 1:
                rows.append(dict(site=name, is_ref=name.startswith("ref_"), radius=r, feature=f, n=len(x), d=cohens_d(x, y)))
rad = pd.DataFrame(rows)

ref = (rad[rad.is_ref].assign(a=lambda t: t.d.abs()).groupby(["radius", "feature"]).a
       .agg(ref_n="count", ref_p90=lambda s: s.quantile(0.9)).reset_index())
sit = rad[~rad.is_ref].assign(a=lambda t: t.d.abs())
out = sit.merge(ref, on=["radius", "feature"], how="left")
out["above"] = (out.a > out.ref_p90) & (out.ref_n >= MIN_REFS)

print(out.pivot_table(index=["site", "feature"], columns="radius", values="d").round(2).to_string())
print("\nfeatures above the random-places p90, out of 10:")
print(out.groupby(["site", "radius"]).above.sum().unstack())
rad.to_csv(f"{OUT}/radius_sweep.csv", index=False)

radius                                100   150   200   300
site           feature                                     
Hampi_baseline BSI                  -0.05 -0.07 -0.04 -0.12
               NDMI                  0.13  0.15  0.11  0.17
               NDVI                  0.12  0.12  0.06  0.04
               NDVI_green_minus_dry  0.39  0.32  0.30  0.24
               SAVI                  0.09  0.14  0.15  0.16
               SWIR1                -0.24 -0.06  0.27  0.24
               VH                    0.78  0.65  0.20 -0.05
               VV                    0.75  0.58  0.21 -0.03
               elev                  0.01 -0.06 -0.16 -0.19
               slope                -0.58 -0.54 -0.32 -0.05
Pattadakal     BSI                  -0.46 -0.29 -0.33 -0.35
               NDMI                  0.29  0.15  0.21  0.29
               NDVI                  0.71  0.63  0.56  0.25
               NDVI_green_minus_dry -0.38 -0.23 -0.14 -0.10
               SAVI                  1.0

In [ ]:
print(out[out.above & (out.radius <= 150)][["site", "radius", "feature", "d", "ref_p90"]].round(2).to_string(index=False))

          site  radius              feature     d  ref_p90
Hampi_baseline     100                   VV  0.75     0.53
Hampi_baseline     100                   VH  0.78     0.54
Hampi_baseline     150                   VV  0.58     0.55
Hampi_baseline     150                   VH  0.65     0.50
    Pattadakal     100                 SAVI  1.01     0.81
    Pattadakal     100                SWIR1  1.22     1.06
    Pattadakal     150                 SAVI  0.84     0.70
    Pattadakal     150                SWIR1  1.14     0.91
       Ramappa     100 NDVI_green_minus_dry -1.08     1.04
       Ramappa     100                   VV -0.74     0.53
       Ramappa     100                   VH -1.22     0.54
       Ramappa     150 NDVI_green_minus_dry -1.24     0.76
       Ramappa     150                   VV -0.69     0.55
       Ramappa     150                   VH -1.11     0.50
